# Additional End of week Exercise - week 2

As required in the week 2 exercise, I built out a question/answer chatbot with the following traits and capabilities.

- Uses GradioUI
- Streaming
- Ability to switch between models
- Ability yo submit questions via audio OR text.
- Response given in audio or text format.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import inspect
print(gr.__version__)
print(gr.__file__)
print(inspect.signature(gr.Chatbot))

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
6.12.0
/Users/cgoose/anaconda3/envs/llms/lib/python3.11/site-packages/gradio/__init__.py
(value: 'list[MessageDict | Message] | Callable | None' = None, *, label: 'str | I18nData | None' = None, every: 'Timer | float | None' = None, inputs: 'Component | Sequence[Component] | set[Component] | None' = None, show_label: 'bool | None' = None, container: 'bool' = True, scale: 'int | None' = None, min_width: 'int' = 160, visible: "bool | Literal['hidden']" = True, elem_id: 'str | None' = None, elem_cl

## Constants / Initial Values

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

OLLAMA_BASE_URL = "http://localhost:11434/v1"

system_prompt = "You are a helpful assistant that can answer questions and help with tasks."



## Initialization

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
openai = OpenAI()
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

MODEL_OPTIONS = {
    "gpt-4o-mini": openai,
    "llama3.2": ollama
}



OpenAI API Key exists and begins sk-proj-


## Chatbot Functions

In [ ]:
# Chat

def chat(message, history, system_message, selected_model):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    stream = MODEL_OPTIONS[selected_model].chat.completions.create(model=selected_model, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": response}
        ], ""

def transcribe_audio(audio_filepath):
    if not audio_filepath:
        return "", None
    with open(audio_filepath, "rb") as audio_file:
        transcript = openai.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )
    
    return transcript.text, None

def text_to_speech(history):    
    final_response = history[-1]["content"][0]["text"]
    speech_file_path = "speech.mp3"
    
    with openai.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=final_response
    ) as response:
        response.stream_to_file(speech_file_path)
    return speech_file_path

## Gradio Chatbot

Chatbot is designed to take in your questions, via text or voice, and spit out an answer in both formats.

In [32]:
with gr.Blocks() as ui:
    with gr.Row():
        gr.Markdown("<h1 style='text-align: center;'>Chatbot</h1>")
    with gr.Group():
        gr.Markdown("<h2>Define your chatbot's system and dropdown</h2>")
        with gr.Row():
            systemPrompt = gr.Textbox(system_prompt, label="Define your chatbot", interactive=True)
            model = gr.Dropdown(choices=MODEL_OPTIONS, value='llama3.2', interactive=True, label="Select your model")
    with gr.Column():
        chatbot = gr.Chatbot()
        with gr.Row():
            audio_output = gr.Audio(label="Audio Output")
        with gr.Group(): 
            with gr.Row():
                with gr.Column():
                    message = gr.Textbox(label="Your Question")
                    submit_btn = gr.Button("Ask")
                with gr.Column():
                    audio_input = gr.Audio(sources="microphone", type="filepath", label="Audio Input")
                    audio_submit = gr.Button("Ask")
        
    submit_btn.click(chat, inputs=[message, chatbot, systemPrompt, model], outputs=[chatbot, message]).then(
        text_to_speech, inputs=[chatbot], outputs=[audio_output]
    )

    audio_submit.click(transcribe_audio, inputs=[audio_input], outputs=[message, audio_input]).then(
        chat, inputs=[message, chatbot, systemPrompt, model], outputs=[chatbot, message]
    ).then(
        text_to_speech, inputs=[chatbot], outputs=[audio_output]
    )

ui.launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7879
* To create a public link, set `share=True` in `launch()`.
